# 02 - Pré-processamento

Este notebook tem duas seções:

1. **Seção 1: Application scoring:** pré-processamento do cadastro (`application_record.csv`) para prever `mau_pagador` com features cadastrais.
2. **Seção 2: Behavioral scoring:** construção de snapshots do histórico de pagamento (`credit_record.csv`) para prever um evento futuro. 

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import RANDOM_STATE, TARGET, DATA_PROCESSED

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", None)

print(f"RANDOM_STATE = {RANDOM_STATE}")
print(f"TARGET = {TARGET}")
print(f"DATA_PROCESSED = {DATA_PROCESSED}")

RANDOM_STATE = 42
TARGET = mau_pagador
DATA_PROCESSED = C:\Users\bruno\Documents\Unisociesc\FIAP_Postech\Fase2\desafio\GitClone\tech-challenge-fase2-grupo17\data\processed


## Seção 1 - Application scoring (cadastro tratado)

O objetivo desta seção é transformar o `df_final.parquet` (produzido no notebook `01_eda.ipynb`) em uma matriz numérica pronta para os classificadores, usando apenas as features cadastrais.

As etapas são:

1. Carregar o `df_final.parquet`.
2. Tratar `DAYS_EMPLOYED` (sentinela 365243 → NaN + flag `esta_desempregado`).
3. Tratar `OCCUPATION_TYPE` (NaN → `"Nao_informado"`; agrupar categorias raras).
4. Criar features derivadas (`cad_idade`, `cad_anos_emprego`, `cad_renda_per_capita`).
5. Aplicar `log1p` na renda.
6. Fazer One-Hot Encoding das categóricas.
7. Criar a coluna `grupo_cadastro` (hash do perfil) para o `GroupKFold`.
8. Salvar em `data/processed/cadastro_tratado.parquet`.

In [2]:
# =============================================================
# 1. Carregar o df_final
# =============================================================

df = pd.read_parquet(DATA_PROCESSED / "df_final.parquet")

print(f"Shape: {df.shape}")
print(f"IDs únicos: {df['ID'].nunique():,}")
print(f"\nColunas:")
print(df.columns.tolist())

Shape: (36457, 34)
IDs únicos: 36,457

Colunas:
['ID', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'FLAG_MOBIL', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS', 'mau_pagador', 'hist_meses_total', 'hist_mes_mais_antigo', 'hist_mes_mais_recente', 'hist_status_max', 'hist_status_medio', 'hist_status_desvio', 'hist_qtd_pago', 'hist_qtd_sem_emprestimo', 'hist_qtd_atraso', 'hist_qtd_atraso_grave', 'hist_prop_pago', 'hist_prop_sem_emprestimo', 'hist_prop_atraso', 'hist_prop_atraso_grave', 'log_renda']


### 2. Tratamento de `DAYS_EMPLOYED`

O valor `365243` é um sentinela para "sem vínculo empregatício ativo" (aposentados, desempregados). <br>
Ele representa **cerca de 16,8%** dos clientes e distorce a distribuição se tratado como dias normais.

**Decisão:**

1. Substituir `365243` por `NaN` em `DAYS_EMPLOYED`.
2. Criar a flag binária `esta_desempregado` (1 se era 365243, 0 caso contrário).
3. Converter `DAYS_EMPLOYED` para anos (`-DAYS_EMPLOYED / 365`) e renomear para `cad_anos_emprego`.

A conversão para anos é monotônica e não altera a informação; apenas facilita a interpretação.

In [3]:
# =============================================================
# 2. Tratamento de DAYS_EMPLOYED
# =============================================================

# 2.1 Flag de desempregado
df["esta_desempregado"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)

# 2.2 Substituir sentinela por NaN
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)

# 2.3 Converter para anos (positivo)
df["cad_anos_emprego"] = -df["DAYS_EMPLOYED"] / 365

# 2.4 Descartar a coluna original
df = df.drop(columns=["DAYS_EMPLOYED"])

# Verificação
print(f"Clientes marcados como desempregados: {df['esta_desempregado'].sum():,} "
      f"({df['esta_desempregado'].mean():.2%})")
print(f"\nEstatísticas de cad_anos_emprego (não-desempregados):")
print(df.loc[df["esta_desempregado"] == 0, "cad_anos_emprego"].describe().round(2))

Clientes marcados como desempregados: 6,135 (16.83%)

Estatísticas de cad_anos_emprego (não-desempregados):
count    30322.00
mean         7.25
std          6.46
min          0.05
25%          2.68
50%          5.46
75%          9.60
max         43.05
Name: cad_anos_emprego, dtype: float64


### 3. Tratamento de `OCCUPATION_TYPE`

`OCCUPATION_TYPE` tem 31% de nulos e 18 categorias. Já discutimos que:

- Os nulos são **estruturais**: 54% são pensionistas e 54% são desempregados (há sobreposição).
- A taxa de mau pagador é **a mesma** entre clientes com e sem ocupação informada (1,68% vs. 1,71%).
- As 7 categorias raras (< 1% dos clientes) têm sinal provavelmente artefato de gêmeos (categorias pequenas infladas por perfis duplicados).

**Decisão:**

1. Substituir `NaN` por `"Nao_informado"`.
2. Agrupar as 7 categorias raras em `"Outros"`.
3. Resultado: 12 categorias (`Nao_informado` + 11 frequentes + `Outros`).

In [4]:
# =============================================================
# 3. Tratamento de OCCUPATION_TYPE
# =============================================================

# 3.1 Preencher NaN com "Nao_informado"
df["OCCUPATION_TYPE"] = df["OCCUPATION_TYPE"].fillna("Nao_informado")

# 3.2 Agrupar categorias raras em "Outros"
categorias_raras = [
    "Private service staff",
    "Low-skill Laborers",
    "Waiters/barmen staff",
    "Secretaries",
    "HR staff",
    "Realty agents",
    "IT staff",
]

df["OCCUPATION_TYPE"] = df["OCCUPATION_TYPE"].where(
    ~df["OCCUPATION_TYPE"].isin(categorias_raras),
    "Outros",
)

# Verificação
print("Distribuição final de OCCUPATION_TYPE:")
print(df["OCCUPATION_TYPE"].value_counts())

Distribuição final de OCCUPATION_TYPE:
OCCUPATION_TYPE
Nao_informado            11323
Laborers                  6211
Core staff                3591
Sales staff               3485
Managers                  3012
Drivers                   2138
High skill tech staff     1383
Accountants               1241
Medicine staff            1207
Outros                    1068
Cooking staff              655
Security staff             592
Cleaning staff             551
Name: count, dtype: int64


### 4. Features derivadas

Três features derivadas do cadastro são criadas:

- **`cad_idade`** = `-DAYS_BIRTH / 365`. Idade em anos, mais intuitiva que dias.
- **`cad_anos_emprego`** já foi criada no tratamento de `DAYS_EMPLOYED`.
- **`cad_renda_per_capita`** = `AMT_INCOME_TOTAL / CNT_FAM_MEMBERS`. Renda familiar por pessoa, que pode capturar pressão financeira melhor que a renda bruta.
- **`log_renda`** já vem do `df_final.parquet` (criada no notebook 01). Mantida.

A decisão de manter `log_renda` (em vez de recriar) é porque ela já está pronta e o `02` é o lugar natural para usá-la.

In [5]:
# =============================================================
# 4. Features derivadas
# =============================================================

# 4.1 Idade em anos
df["cad_idade"] = -df["DAYS_BIRTH"] / 365

# 4.2 Renda per capita
df["cad_renda_per_capita"] = df["AMT_INCOME_TOTAL"] / df["CNT_FAM_MEMBERS"]

# 4.3 Descartar colunas brutas que já foram substituídas
df = df.drop(columns=["DAYS_BIRTH"])

# Verificação
print("Estatísticas das features derivadas:")
print(df[["cad_idade", "cad_anos_emprego", "cad_renda_per_capita", "log_renda"]].describe().round(2))

Estatísticas das features derivadas:
       cad_idade  cad_anos_emprego  cad_renda_per_capita  log_renda
count   36457.00          30322.00              36457.00   36457.00
mean       43.77              7.25             100554.77      12.02
std        11.51              6.46              72678.93       0.48
min        20.52              0.05               5625.00      10.20
25%        34.14              2.68              56250.00      11.71
50%        42.64              5.46              78750.00      11.97
75%        53.25              9.60             123750.00      12.32
max        68.91             43.05             900000.00      14.27


### 5. Coluna `grupo_cadastro` (hash do perfil)

A EDA mostrou que 91,4% dos clientes têm cadastro idêntico a outro. Isso exige um split **por grupo** na modelagem. <br>
Para isso, criamos uma assinatura única de cada perfil, concatenando todas as colunas de cadastro que definem o "gêmeo".

**Importante:** o hash é feito **antes** do encoding das categóricas, para capturar o cadastro original (não as colunas one-hot).

In [6]:
# =============================================================
# 5. Coluna grupo_cadastro (hash do perfil)
# =============================================================

# Colunas que definem o "gêmeo" (cadastro original, sem ID, target e histórico)
colunas_perfil = [
    "CODE_GENDER", "FLAG_OWN_CAR", "FLAG_OWN_REALTY",
    "CNT_CHILDREN", "AMT_INCOME_TOTAL", "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE",
    "FLAG_MOBIL", "FLAG_WORK_PHONE", "FLAG_PHONE", "FLAG_EMAIL",
    "OCCUPATION_TYPE", "CNT_FAM_MEMBERS",
    "esta_desempregado", "cad_idade", "cad_anos_emprego", "cad_renda_per_capita",
]

# Criar hash
df["grupo_cadastro"] = pd.util.hash_pandas_object(
    df[colunas_perfil].astype(str), index=False
).values

# Verificação
print(f"Clientes: {len(df):,}")
print(f"Perfis distintos: {df['grupo_cadastro'].nunique():,}")
print(f"Média de clientes por perfil: {len(df) / df['grupo_cadastro'].nunique():.2f}")

Clientes: 36,457
Perfis distintos: 9,728
Média de clientes por perfil: 3.75


### 6. Encoding das categóricas

As colunas categóricas restantes são:

- `CODE_GENDER`, `FLAG_OWN_CAR`, `FLAG_OWN_REALTY`: 2 categorias cada. Vamos converter para 0/1 diretamente.
- `NAME_INCOME_TYPE`, `NAME_EDUCATION_TYPE`, `NAME_FAMILY_STATUS`, `NAME_HOUSING_TYPE`, `OCCUPATION_TYPE`: mais de 2 categorias. Usamos One-Hot Encoding com `drop_first=True` (para evitar redundância).

**Por que One-Hot e não Label Encoding?**

Label Encoding atribui números arbitrários (ex.: `Working=0`, `Pensioner=1`), introduzindo uma relação de ordem que não existe. Isso engana a Regressão Logística. <br>
One-Hot cria colunas binárias independentes, que é a codificação correta para variáveis nominais.

Para os modelos de árvore (RandomForest, XGBoost), One-Hot também funciona bem.

In [7]:
# =============================================================
# 6. Encoding das categóricas
# =============================================================

# 6.1 Binárias: converter para 0/1
df["CODE_GENDER"] = (df["CODE_GENDER"] == "M").astype(int)
df["FLAG_OWN_CAR"] = (df["FLAG_OWN_CAR"] == "Y").astype(int)
df["FLAG_OWN_REALTY"] = (df["FLAG_OWN_REALTY"] == "Y").astype(int)

# 6.2 Multiclasse: One-Hot Encoding
categoricas_multiclasse = [
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
    "OCCUPATION_TYPE",
]

df = pd.get_dummies(df, columns=categoricas_multiclasse, drop_first=True)

print(f"Shape após encoding: {df.shape}")
print(f"\nColunas resultantes (primeiras 20):")
print(df.columns.tolist()[:20])

Shape após encoding: (36457, 61)

Colunas resultantes (primeiras 20):
['ID', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'FLAG_MOBIL', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL', 'CNT_FAM_MEMBERS', 'mau_pagador', 'hist_meses_total', 'hist_mes_mais_antigo', 'hist_mes_mais_recente', 'hist_status_max', 'hist_status_medio', 'hist_status_desvio', 'hist_qtd_pago', 'hist_qtd_sem_emprestimo']


### 7. Salvar o dataset tratado

O resultado é o `cadastro_tratado.parquet`, com **1 linha por cliente** e todas as features codificadas. <br>
Ele será usado no notebook `03_modelagem.ipynb` para treinar os modelos de application scoring.

A coluna `grupo_cadastro` é preservada para que o `GroupKFold` possa ser usado corretamente na modelagem.

In [18]:
# =============================================================
# 7. Salvar o cadastro_tratado.parquet
# =============================================================

# Separar features, target e grupos
colunas_excluir = ["ID", "grupo_cadastro", TARGET]
colunas_remover = [c for c in df.columns if c.startswith("hist_")] + colunas_excluir

# Salvar o dataset completo (com target e grupo) - mais flexível para o 03
df.to_parquet(DATA_PROCESSED / "cadastro_tratado.parquet", index=False)

print(f"cadastro_tratado.parquet salvo em {DATA_PROCESSED}")
print(f"Shape: {df.shape}")
print(f"Colunas de features: {len([c for c in df.columns if c not in ['ID', 'grupo_cadastro', TARGET]])}")
print(f"Colunas de histórico (a serem descartadas na modelagem de cadastro): "
      f"{len([c for c in df.columns if c.startswith('hist_')])}")

cadastro_tratado.parquet salvo em C:\Users\bruno\Documents\Unisociesc\FIAP_Postech\Fase2\desafio\GitClone\tech-challenge-fase2-grupo17\data\processed
Shape: (36457, 61)
Colunas de features: 58
Colunas de histórico (a serem descartadas na modelagem de cadastro): 14


## Seção 2 - Behavioral scoring (snapshots)

A EDA mostrou que o cadastro quase não tem sinal. As features do histórico (`hist_*`) têm sinal, mas ainda carregam viés de exposição: clientes com histórico mais longo têm mais chances de "ter atrasado algum dia".

Nesta seção, reformulamos o problema para eliminar esse viés:

- **Janela de observação fixa de 6 meses**: as features comportamentais são calculadas apenas com os primeiros 6 meses da conta.
- **Janela de performance fixa de 12 meses**: o target é "atingiu atraso ≥ 60 dias em algum dos meses 7 a 18".
- **Exclusão de quem já estava em atraso grave na observação**: prever que quem já está inadimplente continuará inadimplente não é previsão.
- **Mesma exposição para todos**: todos os clientes com pelo menos 18 meses de histórico entram com a mesma janela. Clientes com histórico menor são descartados.

O resultado é um dataset de snapshots, onde cada linha é "um cliente observado por 6 meses, prevendo o comportamento dos 12 meses seguintes".

In [9]:
# =============================================================
# 1. Carregar o credit_record e definir janelas
# =============================================================

from src.data import load_credit

cred = load_credit()

# Parâmetros das janelas
JANELA_OBS = 6     # meses de observação (features)
JANELA_PERF = 12   # meses de performance (target)

print(f"credit_record: {cred.shape}")
print(f"Clientes únicos: {cred['ID'].nunique():,}")
print(f"Janela de observação: {JANELA_OBS} meses")
print(f"Janela de performance: {JANELA_PERF} meses")

credit_record: (1048575, 3)
Clientes únicos: 45,985
Janela de observação: 6 meses
Janela de performance: 12 meses


### 1. Preparar o `credit_record` para a lógica de snapshots

O `credit_record` tem `MONTHS_BALANCE` em ordem decrescente (0 é o mês mais recente, -1 é o mês anterior, etc.). <br>
Para a lógica de snapshots, é mais fácil trabalhar com uma coluna `mes` que conta do início da conta:

- `mes = 0` é o primeiro mês da conta.
- `mes = n_meses - 1` é o último mês observado.

Também mapeamos `STATUS` para um valor numérico de severidade (mesma convenção do notebook 01).

In [10]:
# =============================================================
# 2. Preparar o credit_record
# =============================================================

# Mapeamento de STATUS para numérico (mesma convenção do 01_eda)
MAPA_STATUS = {
    "C": 0, "X": 0, "0": 0,
    "1": 1, "2": 2, "3": 3, "4": 4, "5": 5,
}

cred["status_num"] = cred["STATUS"].map(MAPA_STATUS)

# Conta os meses por cliente (do início da conta até o mês atual)
cred = cred.sort_values(["ID", "MONTHS_BALANCE"])
cred["mes"] = cred.groupby("ID")["MONTHS_BALANCE"].transform(
    lambda s: s - s.min()
)

# Quantos meses cada cliente tem no total
n_meses_por_cliente = cred.groupby("ID")["mes"].max() + 1

print(f"Meses de histórico por cliente:")
print(n_meses_por_cliente.describe().round(1))
print(f"\nClientes com pelo menos {JANELA_OBS + JANELA_PERF} meses: "
      f"{(n_meses_por_cliente >= JANELA_OBS + JANELA_PERF).sum():,}")

Meses de histórico por cliente:
count    45985.0
mean        22.8
std         15.5
min          1.0
25%         10.0
50%         19.0
75%         34.0
max         61.0
Name: mes, dtype: float64

Clientes com pelo menos 18 meses: 25,047


### 2. Construir os snapshots (janela fixa)

Para cada cliente com pelo menos 18 meses de histórico:

1. **Observação (meses 0 a 5)**: calculamos features comportamentais.
2. **Exclusão**: se o cliente já atingiu `status_num >= 2` em algum mês da observação, ele é descartado (não faz sentido prever "vai atrasar" para quem já está atrasado).
3. **Performance (meses 6 a 17)**: o target é 1 se o cliente atingiu `status_num >= 2` em algum mês da performance, 0 caso contrário.

In [11]:
# =============================================================
# 3. Construir os snapshots (janela fixa)
# =============================================================

# Filtrar clientes com pelo menos JANELA_OBS + JANELA_PERF meses
clientes_elegiveis = n_meses_por_cliente[
    n_meses_por_cliente >= JANELA_OBS + JANELA_PERF
].index

cred_eleg = cred[cred["ID"].isin(clientes_elegiveis)].copy()

# Separar observação e performance
obs = cred_eleg[cred_eleg["mes"] < JANELA_OBS].copy()
perf = cred_eleg[
    (cred_eleg["mes"] >= JANELA_OBS) &
    (cred_eleg["mes"] < JANELA_OBS + JANELA_PERF)
].copy()

print(f"Registros na observação: {len(obs):,}")
print(f"Registros na performance: {len(perf):,}")

# -----------------------------------------------------------------
# Features comportamentais (por cliente, na janela de observação)
# -----------------------------------------------------------------
obs_agg = obs.groupby("ID").agg(
    # Severidade
    obs_status_max      = ("status_num", "max"),
    obs_status_medio    = ("status_num", "mean"),
    obs_status_desvio   = ("status_num", "std"),

    # Contagens por tipo de status
    obs_qtd_pago        = ("STATUS", lambda s: (s == "C").sum()),
    obs_qtd_sem_credito = ("STATUS", lambda s: (s == "X").sum()),
    obs_qtd_atraso      = ("status_num", lambda s: (s >= 1).sum()),
    obs_qtd_atraso_30   = ("status_num", lambda s: (s >= 2).sum()),
).reset_index()

# -----------------------------------------------------------------
# Features derivadas
# -----------------------------------------------------------------
obs_agg["obs_frac_pago"]          = obs_agg["obs_qtd_pago"] / JANELA_OBS
obs_agg["obs_frac_sem_credito"]   = obs_agg["obs_qtd_sem_credito"] / JANELA_OBS
obs_agg["obs_frac_atraso"]        = obs_agg["obs_qtd_atraso"] / JANELA_OBS
obs_agg["obs_frac_atraso_30"]     = obs_agg["obs_qtd_atraso_30"] / JANELA_OBS

# Tendência: segunda metade da observação vs. primeira metade
obs_metade1 = obs[obs["mes"] < JANELA_OBS // 2].groupby("ID").agg(
    obs_frac_atraso_1a_metade = ("status_num", lambda s: (s >= 1).mean())
)
obs_metade2 = obs[obs["mes"] >= JANELA_OBS // 2].groupby("ID").agg(
    obs_frac_atraso_2a_metade = ("status_num", lambda s: (s >= 1).mean())
)
obs_agg = obs_agg.merge(obs_metade1, on="ID", how="left")
obs_agg = obs_agg.merge(obs_metade2, on="ID", how="left")
obs_agg["obs_tendencia"] = (
    obs_agg["obs_frac_atraso_2a_metade"] - obs_agg["obs_frac_atraso_1a_metade"]
)

# Status no último mês da observação
obs_ultimo = (
    obs.sort_values("mes")
    .groupby("ID")
    .tail(1)
    .set_index("ID")["status_num"]
    .rename("obs_status_ultimo_mes")
)
obs_agg = obs_agg.merge(obs_ultimo, on="ID", how="left")

# Desvio-padrão é NaN quando só há 1 mês -> 0
obs_agg["obs_status_desvio"] = obs_agg["obs_status_desvio"].fillna(0)

# -----------------------------------------------------------------
# Excluir clientes que já estavam em atraso ≥ 60 dias na observação
# -----------------------------------------------------------------
n_antes = len(obs_agg)
obs_agg = obs_agg[obs_agg["obs_status_max"] < 2].copy()
n_depois = len(obs_agg)
print(f"\nClientes excluídos por já estarem em atraso ≥ 60 dias na observação: "
      f"{n_antes - n_depois:,}")

# -----------------------------------------------------------------
# Target (performance)
# -----------------------------------------------------------------
perf_target = perf.groupby("ID").agg(
    target_snapshot = ("status_num", lambda s: (s >= 2).any().astype(int))
).reset_index()

# -----------------------------------------------------------------
# Montar o snapshot final
# -----------------------------------------------------------------
snapshots = obs_agg.merge(perf_target, on="ID", how="inner")

print(f"\nSnapshots finais: {len(snapshots):,}")
print(f"Taxa de mau pagador (target_snapshot): {snapshots['target_snapshot'].mean():.2%}")

Registros na observação: 150,282
Registros na performance: 300,564

Clientes excluídos por já estarem em atraso ≥ 60 dias na observação: 136

Snapshots finais: 24,911
Taxa de mau pagador (target_snapshot): 1.18%


### 3. Análise rápida do snapshot

Antes de salvar, vale olhar a distribuição do target e as estatísticas das features. O objetivo é confirmar que o snapshot ficou coerente.

In [12]:
# =============================================================
# 4. Análise rápida do snapshot
# =============================================================

print("Distribuição do target:")
print(snapshots["target_snapshot"].value_counts())
print()
print(f"Taxa de mau pagador: {snapshots['target_snapshot'].mean():.2%}")
print()

print("Estatísticas das features comportamentais:")
features_obs = [c for c in snapshots.columns if c.startswith("obs_")]
print(snapshots[features_obs].describe().round(3).T)

Distribuição do target:
target_snapshot
0    24618
1      293
Name: count, dtype: int64

Taxa de mau pagador: 1.18%

Estatísticas das features comportamentais:
                             count   mean    std    min  25%  50%    75%  \
obs_status_max             24911.0  0.052  0.221  0.000  0.0  0.0  0.000   
obs_status_medio           24911.0  0.013  0.062  0.000  0.0  0.0  0.000   
obs_status_desvio          24911.0  0.023  0.099  0.000  0.0  0.0  0.000   
obs_qtd_pago               24911.0  0.388  1.083  0.000  0.0  0.0  0.000   
obs_qtd_sem_credito        24911.0  1.595  2.256  0.000  0.0  0.0  2.000   
obs_qtd_atraso             24911.0  0.076  0.374  0.000  0.0  0.0  0.000   
obs_qtd_atraso_30          24911.0  0.000  0.000  0.000  0.0  0.0  0.000   
obs_frac_pago              24911.0  0.065  0.180  0.000  0.0  0.0  0.000   
obs_frac_sem_credito       24911.0  0.266  0.376  0.000  0.0  0.0  0.333   
obs_frac_atraso            24911.0  0.013  0.062  0.000  0.0  0.0  0.000   
obs_

### 4. Salvar o `snapshots.parquet`

O resultado é o `snapshots.parquet`, com **1 linha por snapshot** (neste caso, 1 linha por cliente, porque estamos usando janela fixa). <br>
Ele será usado no `03_modelagem.ipynb` para treinar os modelos de behavioral scoring.

A próxima seção fará o merge com o cadastro (features cadastrais + features comportamentais).

In [13]:
# =============================================================
# 5. Salvar o snapshots.parquet
# =============================================================

snapshots.to_parquet(DATA_PROCESSED / "snapshots.parquet", index=False)

print(f"snapshots.parquet salvo em {DATA_PROCESSED}")
print(f"Shape: {snapshots.shape}")
print(f"Colunas: {snapshots.columns.tolist()}")

snapshots.parquet salvo em C:\Users\bruno\Documents\Unisociesc\FIAP_Postech\Fase2\desafio\GitClone\tech-challenge-fase2-grupo17\data\processed
Shape: (24911, 17)
Colunas: ['ID', 'obs_status_max', 'obs_status_medio', 'obs_status_desvio', 'obs_qtd_pago', 'obs_qtd_sem_credito', 'obs_qtd_atraso', 'obs_qtd_atraso_30', 'obs_frac_pago', 'obs_frac_sem_credito', 'obs_frac_atraso', 'obs_frac_atraso_30', 'obs_frac_atraso_1a_metade', 'obs_frac_atraso_2a_metade', 'obs_tendencia', 'obs_status_ultimo_mes', 'target_snapshot']


## Seção 2.3 - Merge do snapshot com cadastro

O `snapshots.parquet` tem apenas as features comportamentais. Para testar se o cadastro acrescenta informação ao behavioral scoring, fazemos o merge com o `cadastro_tratado.parquet`.

O resultado é o `snapshots_com_cadastro.parquet`, que contém:

- Features comportamentais (`obs_*`)
- Features cadastrais (já codificadas)
- Target (`target_snapshot`)
- Coluna `grupo_cadastro` para o `GroupKFold`

In [14]:
# =============================================================
# 6. Merge do snapshot com cadastro
# =============================================================

# Carregar o cadastro tratado
cadastro = pd.read_parquet(DATA_PROCESSED / "cadastro_tratado.parquet")

# Carregar o snapshot
snapshots = pd.read_parquet(DATA_PROCESSED / "snapshots.parquet")

print(f"Cadastro tratado: {cadastro.shape}")
print(f"Snapshots: {snapshots.shape}")

# Merge 1:1 por ID
snapshots_com_cadastro = snapshots.merge(
    cadastro.drop(columns=[TARGET]),   # não trazer o mau_pagador do cadastro
    on="ID",
    how="inner",
    validate="one_to_one",
)

print(f"\nSnapshots com cadastro: {snapshots_com_cadastro.shape}")
print(f"IDs únicos: {snapshots_com_cadastro['ID'].nunique():,}")

Cadastro tratado: (36457, 61)
Snapshots: (24911, 17)

Snapshots com cadastro: (18340, 76)
IDs únicos: 18,340


### 6.1 Análise do merge

O merge reduziu de **24.911** para **18.340** snapshots. A diferença de 6.571 clientes (26%) corresponde a clientes que têm histórico de pagamento em `credit_record.csv` mas **não têm cadastro** em `application_record.csv`. <br>
Esse é o **viés de seleção** discutido no notebook 01: o `credit_record.csv` cobre apenas clientes que já tiveram cartão emitido, e muitos deles não estão no cadastro. É uma limitação dos dados, não do merge.

**Decisão:** manter os dois datasets em paralelo:

- `snapshots.parquet` (24.911 clientes): behavioral scoring puro.
- `snapshots_com_cadastro.parquet` (18.340 clientes): behavioral scoring enriquecido com cadastro.

O `03_modelagem` vai treinar modelos nos dois e comparar.

In [15]:
# =============================================================
# 7. Verificação do merge
# =============================================================

# IDs únicos (não deve duplicar)
assert snapshots_com_cadastro["ID"].is_unique, \
    "Merge duplicou IDs (NÃO deveria acontecer)!"

# Quantificar a perda
n_antes = len(snapshots)
n_depois = len(snapshots_com_cadastro)
perdidos = n_antes - n_depois

print(f"Snapshots antes do merge: {n_antes:,}")
print(f"Snapshots após o merge:   {n_depois:,}")
print(f"Snapshots perdidos:       {perdidos:,} ({perdidos / n_antes:.2%})")
print(f"Motivo: clientes sem cadastro em application_record.csv")
print()

# Distribuição do target (não deve ter mudado)
print(f"Taxa de mau pagador (target_snapshot): "
      f"{snapshots_com_cadastro['target_snapshot'].mean():.2%}")

# Nulos
nulos = snapshots_com_cadastro.isna().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)

print(f"\nColunas com nulos:")
if len(nulos) == 0:
    print("  (nenhum)")
else:
    print(nulos)

Snapshots antes do merge: 24,911
Snapshots após o merge:   18,340
Snapshots perdidos:       6,571 (26.38%)
Motivo: clientes sem cadastro em application_record.csv

Taxa de mau pagador (target_snapshot): 1.43%

Colunas com nulos:
cad_anos_emprego    2946
dtype: int64


### 7.1 Verificação dos grupos

A coluna `grupo_cadastro` foi herdada do `cadastro_tratado.parquet`. <br>
Vale verificar que ela está coerente: os snapshots devem ter menos grupos únicos do que linhas, porque cada cliente compartilha grupo com seus gêmeos.

In [16]:
# =============================================================
# 8. Verificação dos grupos
# =============================================================

n_linhas = len(snapshots_com_cadastro)
n_grupos = snapshots_com_cadastro["grupo_cadastro"].nunique()

print(f"Snapshots: {n_linhas:,}")
print(f"Grupos distintos: {n_grupos:,}")
print(f"Média de snapshots por grupo: {n_linhas / n_grupos:.2f}")
print(f"Maior grupo: {snapshots_com_cadastro.groupby('grupo_cadastro').size().max()}")

# Checar se há gêmeos com targets diferentes
grupos_por_target = snapshots_com_cadastro.groupby("grupo_cadastro")[
    "target_snapshot"
].nunique()
n_conflitantes = (grupos_por_target > 1).sum()

print(f"\nGrupos com target conflitante (gêmeos com rótulos diferentes): {n_conflitantes:,}")

Snapshots: 18,340
Grupos distintos: 6,694
Média de snapshots por grupo: 2.74
Maior grupo: 19

Grupos com target conflitante (gêmeos com rótulos diferentes): 100


### 8. Salvar o `snapshots_com_cadastro.parquet`

O resultado final desta seção é o `snapshots_com_cadastro.parquet`, que contém comportamento + cadastro + target. <br>
Ele será usado no notebook `03_modelagem.ipynb` para treinar os modelos de behavioral scoring enriquecidos.

In [17]:
# =============================================================
# 9. Salvar o snapshots_com_cadastro.parquet
# =============================================================

snapshots_com_cadastro.to_parquet(
    DATA_PROCESSED / "snapshots_com_cadastro.parquet",
    index=False,
)

print(f"snapshots_com_cadastro.parquet salvo em {DATA_PROCESSED}")
print(f"Shape: {snapshots_com_cadastro.shape}")
print(f"\nDataset final do 02:")
print(f"  - Snapshots: {n_linhas:,}")
print(f"  - Features: {snapshots_com_cadastro.shape[1] - 3} (sem ID, target e grupo)")
print(f"  - Taxa de mau pagador: {snapshots_com_cadastro['target_snapshot'].mean():.2%}")

snapshots_com_cadastro.parquet salvo em C:\Users\bruno\Documents\Unisociesc\FIAP_Postech\Fase2\desafio\GitClone\tech-challenge-fase2-grupo17\data\processed
Shape: (18340, 76)

Dataset final do 02:
  - Snapshots: 18,340
  - Features: 73 (sem ID, target e grupo)
  - Taxa de mau pagador: 1.43%
